In [0]:
%pip install dbldatagen

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Generate 1000 base rows
df_base = spark.range(1000)

raw_sales = df_base.select(
    F.concat(F.lit("SAL-"), F.expr("uuid()")).alias("sale_id"),
    # MESS: 15% Nulls and some empty strings
    F.when(F.rand() < 0.15, F.lit(None)).when(F.rand() < 0.05, F.lit("")).otherwise(F.concat(F.lit("DLR-"), F.expr("floor(rand()*100 + 1000)"))).alias("dealer_id"),
    # MESS: Mixed date formats and invalid strings
    F.when(F.rand() < 0.25, F.lit("2026/03/15"))
     .when(F.rand() < 0.25, F.lit("15-03-2026"))
     .when(F.rand() < 0.25, F.lit("2026.03.15"))
     .when(F.rand() < 0.10, F.lit("UNKNOWN"))
     .otherwise(F.lit("2026-03-15")).alias("tx_date"),
    # NESTED STRUCT: Equipment Info
    F.struct(
        F.expr("floor(rand()*50)").alias("m_id"),
        (F.rand() * 100000).alias("price")
    ).alias("equip"),
    # ARRAY OF STRUCTS: Incentive Programs
    F.array(
        F.struct(F.lit("REBATE_Q1").alias("p_name"), F.lit(0.05).alias("perc")),
        F.struct(F.lit("DEALER_PROMO").alias("p_name"), F.lit(0.02).alias("perc"))
    ).alias("progs")
)

raw_sales.write.mode("overwrite").format("json").save("s3://john-deere-ro/raw_incentive_sales/")

In [0]:
dealers_data = [
    ("DLR-1001", "  John Deere Pune  ", "WEST", "Gold"),
    ("DLR-1001", "John Deere Pune", "WEST", "Silver"), # DUPLICATE with lower tier
    ("DLR-1002", "Moline Tractors Ltd  ", "NORTH", "Silver"),
    ("DLR-1003", "  Kochi Agri Hub", "SOUTH", "Bronze"),
    ("DLR-1004", "Punjab Farm Equip", "NORTH", "Standard"),
    ("DLR-1005", "Mysore Dealers", "SOUTH", None) # NULL Tier
]

df_dealers = spark.createDataFrame(dealers_data, ["dealer_id", "dealer_name", "region", "tier"])
df_dealers.write.mode("overwrite").format("delta").save("s3://john-deere-ro/dim_dealers_raw/")

In [0]:
raw_equip = spark.range(50).select(
    F.concat(F.lit("MOD-"), F.col("id")).alias("model_id"),
    # MESS: Inconsistent Casing
    F.when(F.rand() < 0.5, F.lit("TRACTOR")).otherwise(F.lit("tractor")).alias("category"),
    # MESS: Negative base prices
    F.when(F.rand() < 0.1, F.lit(-999.99)).otherwise(F.rand() * 80000).alias("base_price"),
    F.lit(True).alias("is_active")
)

raw_equip.write.mode("overwrite").format("delta").save("s3://john-deere-ro/dim_equipment_raw/")

In [0]:
raw_payouts = spark.range(500).select(
    F.concat(F.lit("PAY-"), F.expr("uuid()")).alias("payout_id"),
    F.concat(F.lit("SAL-"), F.expr("floor(rand()*1000)")).alias("sale_id"),
    # MESS: Status is SUCCESS but amount is 0
    F.when(F.rand() < 0.1, F.lit(0.0)).otherwise(F.rand() * 5000).alias("amount_paid"),
    F.when(F.col("amount_paid") == 0, F.lit("SUCCESS")).otherwise(F.lit("PENDING")).alias("payout_status")
)

raw_payouts.write.mode("overwrite").format("delta").save("s3://john-deere-ro/legacy_payouts_mysql/")